# Colab Training Notebook

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
!pip install -q torchmetrics mlflow tqdm

In [4]:
import torch
import shutil, os, glob
import mlflow
import pandas as pd
import sys
import argparse
from dotenv import load_dotenv
from pathlib import Path

In [5]:
load_dotenv()

In [6]:
# Check GPU
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU")

In [7]:
# Set Drive path
DRIVE_PATH = Path(os.environ["DRIVE_PATH"])
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/splits', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/labeled', exist_ok=True)
print("Drive path ready")

In [14]:
if str(DRIVE_PATH) not in sys.path:
    sys.path.append(str(DRIVE_PATH))

from src.train import main as train_main
from src.evaluate import evaluate 

In [8]:
# Set MLflow tracking URI to local SQLite database
mlflow.set_tracking_uri(f"sqlite:///{DRIVE_PATH}/mlflow.db")
mlflow.set_experiment("visiomark-image-model")
print("MLflow ready")

In [10]:
# Check data splits
splits = {}
for split in ['train', 'val', 'test']:
    path = f"{DRIVE_PATH}/data/splits/{split}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        splits[split] = df
        print(f"{split}: {len(df)} samples")
        print(f"content_type distribution: {df['content_type_label'].value_counts().to_dict()}")
        print(f"mood distribution:         {df['mood_label'].value_counts().to_dict()}")
        print()
    else:
        print(f"{split}.csv not found")

### Phase 1

In [10]:
print("Starting Phase 1.")
print("=" * 60)

args_phase1 = argparse.Namespace(
    phase=1,
    epochs=15,
    batch_size=16,
    lr=1e-3,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=None,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

checkpoints_path = train_main(args_phase1)
print("Phase 1 checkpoints saved to: /checkpoints ")

### Phase 2

In [11]:
print("Starting Phase 2.")
print("=" * 60)

phase1_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase1*.pt'))

if not phase1_checkpoints:
    raise FileNotFoundError("No Phase 1 checkpoint found.")

best_phase1 = phase1_checkpoints[-1]

args_phase2 = argparse.Namespace(
    phase=2,
    epochs=50,
    batch_size=16,
    lr=1e-5,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=best_phase1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

final_checkpoints_path = train_main(args_phase2)
print("Phase 2 checkpoints saved to: /checkpoints ")

In [12]:
# Copy MLflow DB if using local mode
db_source = f"{DRIVE_PATH}/mlflow.db"
db_backup = f"{DRIVE_PATH}/mlflow_backup.db"

if os.path.exists(db_source):
    shutil.copy2(db_source, db_backup)
    print("MLflow database backed up saved.")
else:
    print("MLflow DB not found.")

print("\nALL TRAINING COMPLETE!")

## Evaluation

In [11]:
phase2_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase2*.pt'))

In [12]:
checkpoint_path = phase2_checkpoints[-1] if phase2_checkpoints else None
if checkpoint_path is None:
    raise FileNotFoundError("No Phase 2 checkpoint found for evaluation.")
print("Checkpoint ready for evaluation")

In [ ]:
result = evaluate(
    checkpoint=checkpoint_path,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    test_csv=f"{DRIVE_PATH}/data/splits/test.csv",
    save_path=f"{DRIVE_PATH}/eval/final_results.json",
)
print("Results saved to eval/final_results.json")